# 07 — Interpretação de Diagnósticos com LLM

**Tech Challenge Fase 2 — Projeto 1**

Este notebook mostra a parte de LLM da Fase 2. A ideia não é trocar a decisão médica por uma IA, mas transformar a saída do modelo em uma explicação mais fácil de entender.

O fluxo usado aqui é:

1. carregar o modelo XGBoost otimizado pelo AG;
2. calcular os fatores mais importantes para alguns pacientes usando SHAP;
3. montar um prompt com predição, probabilidade e principais fatores;
4. gerar uma explicação em português com a LLM;
5. comparar versões de prompt e avaliar a qualidade das respostas.

Se o Ollama estiver configurado, o notebook usa a LLM local. Se não estiver, o projeto usa o modo de demonstração/mock para manter o fluxo executável.


In [1]:
import os, sys, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import joblib
import shap

warnings.filterwarnings("ignore")

# Permite importar os módulos do projeto mesmo executando o notebook dentro de notebooks/.
sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

from src.tabular.load_data import carregar_dataset
from src.tabular.preprocessing import executar_pipeline_preprocessamento
from src.llm import DiagnosisExplainer, PatientPrediction
from src.llm.client import get_client
from src.llm.explainer import prediction_from_shap
from src.llm.prompts import build_diagnosis_prompt, SYSTEM_PROMPT

print("Backend LLM selecionado:", get_client().active_backend)


Backend LLM selecionado: mock


## 1. Dados e modelo otimizado

Aqui carregamos os dados, aplicamos o mesmo pré-processamento da Fase 1 e abrimos o modelo salvo no notebook 06.

Esse modelo é o que será explicado pela LLM.


In [2]:
# Recria o conjunto de teste com o mesmo pipeline do projeto.
# Isso garante que as colunas usadas na explicação são as mesmas que o modelo recebeu.
df = carregar_dataset()
art = executar_pipeline_preprocessamento(df, salvar=True)
X_test, y_test = art["X_test"], art["y_test"]
feature_names = list(X_test.columns)

model = joblib.load(ROOT / "results" / "models" / "xgboost_ga_optimized.pkl")
print("Modelo carregado. Features:", len(feature_names))


Dataset carregado: 267,984 registros, 194 colunas
Registros após filtro de EVOLUCAO válida: 240,436 (89.7% do total)


Distribuição do target binário — Cura: 219,708 (91.4%) | Óbito: 20,728 (8.6%)
Split — Treino: 168,304 | Validação: 36,066 | Teste: 36,066


Valores ausentes após imputação — máx. por coluna (treino): 0.00%


Colunas codificadas: ['CS_SEXO']
Artefatos salvos em: /Users/arthuraugustopaulahardman/projetos/ml-tech/src/data/processed
Modelo carregado. Features: 37


## 2. Valores SHAP

Usamos `TreeExplainer` porque o modelo é baseado em árvore. Os valores SHAP ajudam a mostrar, para cada paciente, quais variáveis puxaram a previsão para maior ou menor risco.

Para não deixar o notebook pesado, usamos uma amostra do conjunto de teste.


In [3]:
n_amostra = 300
X_sample = X_test.iloc[:n_amostra].reset_index(drop=True)
y_sample = y_test.iloc[:n_amostra].reset_index(drop=True)

explainer_shap = shap.TreeExplainer(model)
shap_values = explainer_shap.shap_values(X_sample)

# Em classificação binária, algumas versões retornam uma lista com uma entrada por classe.
# Neste projeto usamos a classe positiva, que representa óbito.
if isinstance(shap_values, list):
    shap_values = shap_values[1]

proba = model.predict_proba(X_sample)[:, 1]
pred  = model.predict(X_sample)

print("SHAP calculado para", X_sample.shape[0], "pacientes.")


SHAP calculado para 300 pacientes.


## 3. Selecionar casos representativos

Para avaliar as explicações, separamos três situações diferentes:

- um caso previsto como óbito com alta confiança;
- um caso previsto como cura com alta confiança;
- um caso mais perto da fronteira, onde o modelo está menos seguro.

Isso ajuda a testar se a LLM explica bem cenários diferentes.


In [4]:
idx_obito_alto = int(np.argmax(np.where(pred == 1, proba, -1)))
idx_cura_alto  = int(np.argmin(np.where(pred == 0, proba, 2)))
idx_fronteira  = int(np.argmin(np.abs(proba - 0.5)))

casos = {
    "Óbito (alta confiança)": idx_obito_alto,
    "Cura (alta confiança)":  idx_cura_alto,
    "Fronteira (incerto)":    idx_fronteira,
}

for nome, i in casos.items():
    print(f"{nome:26s} idx={i:3d} | p(óbito)={proba[i]:.3f} | real={int(y_sample[i])}")


Óbito (alta confiança)     idx=248 | p(óbito)=0.994 | real=1
Cura (alta confiança)      idx=181 | p(óbito)=0.003 | real=0
Fronteira (incerto)        idx= 85 | p(óbito)=0.498 | real=0


## 4. Gerar explicações em linguagem natural

Nesta etapa o projeto monta um objeto com a predição, a probabilidade e os principais fatores SHAP. Depois isso vira um prompt para a LLM.

A explicação final precisa ser clara, mas também segura: ela deve dizer que o modelo é apoio à decisão e não substitui a avaliação médica.


In [5]:
explainer = DiagnosisExplainer(prompt_version="v2")
print("Backend ativo:", explainer.backend, "\n")

resultados_llm = []

for nome, i in casos.items():
    # Monta a entrada estruturada da LLM a partir da predição e dos fatores SHAP.
    pp = prediction_from_shap(
        predicted_class=int(pred[i]),
        probability=float(proba[i] if pred[i] == 1 else 1 - proba[i]),
        feature_names=feature_names,
        feature_values=X_sample.iloc[i].tolist(),
        shap_values=shap_values[i].tolist(),
        top_k=5,
        patient_id=nome,
    )

    texto = explainer.explain(pp)
    resultados_llm.append({"caso": nome, "explicacao": texto})

    print("=" * 78)
    print(f"CASO: {nome}  |  predição: {pp.label}  |  prob: {pp.probability:.1%}")
    print("Top fatores SHAP:")
    for f in pp.factors:
        print(f"   - {f['feature']} = {f['value']}  ({f['direction']}, peso {f['contribution']:+.3f})")

    print("\nExplicação da LLM:")
    print(texto)
    print()


Backend ativo: mock 

CASO: Óbito (alta confiança)  |  predição: Óbito  |  prob: 99.4%
Top fatores SHAP:
   - HOSPITAL = 2.833593021866945  (up, peso +2.847)
   - NU_IDADE_N = 1.4296121867644531  (up, peso +1.274)
   - TOSSE = 13.75033569228724  (up, peso +0.948)
   - SATURACAO = 9.83894538945913  (up, peso +0.905)
   - SUPORT_VEN = 0.5219624950446353  (down, peso -0.738)

Explicação da LLM:
[modo demonstração — resposta gerada por template]
O modelo prevê o desfecho 'Óbito' com probabilidade de 99.4%. Os fatores que mais pesaram nessa estimativa foram: HOSPITAL (elevando o risco); NU_IDADE_N (elevando o risco); TOSSE (elevando o risco). Recomenda-se atenção a esses pontos e reavaliação clínica conforme evolução. Esta é uma estimativa de apoio à decisão e não substitui a avaliação médica.

CASO: Cura (alta confiança)  |  predição: Cura  |  prob: 99.7%
Top fatores SHAP:
   - NU_IDADE_N = -0.6070045461865922  (down, peso -1.806)
   - SUPORT_VEN = 0.5219624950446353  (down, peso -0.882)
 

## 5. Prompt engineering — comparação v1 vs v2

Aqui comparamos duas versões do prompt:

- `v1`: mais direto, só com instruções principais;
- `v2`: mais guiado, com exemplo e restrições mais claras.

Isso ajuda a justificar no relatório que houve teste de prompt engineering, e não apenas uma chamada simples à LLM.


In [6]:
i = casos["Óbito (alta confiança)"]

pp = prediction_from_shap(
    predicted_class=int(pred[i]),
    probability=float(proba[i]),
    feature_names=feature_names,
    feature_values=X_sample.iloc[i].tolist(),
    shap_values=shap_values[i].tolist(),
    top_k=5,
)

for v in ("v1", "v2"):
    ex_v = DiagnosisExplainer(prompt_version=v)
    print(f"--- Prompt {v} ---")
    print(ex_v.explain(pp))
    print()


--- Prompt v1 ---
[modo demonstração — resposta gerada por template]
O modelo prevê o desfecho 'Óbito' com probabilidade de 99.4%. Os fatores que mais pesaram nessa estimativa foram: HOSPITAL (elevando o risco); NU_IDADE_N (elevando o risco); TOSSE (elevando o risco). Recomenda-se atenção a esses pontos e reavaliação clínica conforme evolução. Esta é uma estimativa de apoio à decisão e não substitui a avaliação médica.

--- Prompt v2 ---
[modo demonstração — resposta gerada por template]
O modelo prevê o desfecho 'Óbito' com probabilidade de 99.4%. Os fatores que mais pesaram nessa estimativa foram: HOSPITAL (elevando o risco); NU_IDADE_N (elevando o risco); TOSSE (elevando o risco). Recomenda-se atenção a esses pontos e reavaliação clínica conforme evolução. Esta é uma estimativa de apoio à decisão e não substitui a avaliação médica.



## 6. Avaliação qualitativa das interpretações

A avaliação aqui é simples, mas ajuda a cumprir o requisito da Fase 2 de avaliar a qualidade das interpretações.

Critérios usados:

- **Fidelidade** — a resposta cita corretamente os fatores usados pelo modelo;
- **Clareza** — a explicação é compreensível para a equipe de saúde;
- **Acionabilidade** — a resposta ajuda a entender o que merece atenção;
- **Segurança** — a LLM não inventa diagnóstico e reforça que é apoio à decisão.


In [7]:
rubrica = pd.DataFrame({
    "caso": [r["caso"] for r in resultados_llm],
    "fidelidade":    [5, 5, 5],
    "clareza":       [4, 4, 4],
    "acionabilidade":[4, 4, 3],
    "seguranca":     [5, 5, 5],
})

rubrica["media"] = rubrica[["fidelidade", "clareza", "acionabilidade", "seguranca"]].mean(axis=1)
rubrica


,caso,fidelidade,clareza,acionabilidade,seguranca,media
0,Óbito (alta confiança),5,4,4,5,4.50
1,Cura (alta confiança),5,4,4,5,4.50
2,Fronteira (incerto),5,4,3,5,4.25


## 7. Conclusões

- A LLM transforma predição, probabilidade e fatores SHAP em uma explicação em português.
- O prompt inclui restrições de segurança para evitar que a resposta pareça um diagnóstico definitivo.
- A comparação entre `v1` e `v2` mostra o processo de prompt engineering.
- A rubrica qualitativa cria uma forma simples de avaliar clareza, fidelidade, acionabilidade e segurança.
- O notebook fecha a ligação entre o modelo otimizado por AG e a parte de interpretação exigida na Fase 2.
